In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_blobs
import re
from collections import Counter

# Convert TF-IDF dictionaries to vectors
from sklearn.feature_extraction import DictVectorizer

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("abhi8923shriv/sentiment-analysis-dataset")

print("Path to dataset files:", path)

Path to dataset files: /home/rito/.cache/kagglehub/datasets/abhi8923shriv/sentiment-analysis-dataset/versions/9


In [4]:
path = path + "/train.csv"
data = pd.read_csv(path, encoding='latin1')
df = pd.DataFrame(data)

In [5]:
sentiment_map = {
    'negative': 0,
    'neutral': 1,
    'positive': 1
}

df['sentiment_label'] = df['sentiment'].map(sentiment_map)

In [6]:
time_map = {
    'morning': [1,0,0],
    'noon': [0,1,0],
    'night': [0,0,1]
}

df['time_label'] = df['Time of Tweet'].map(time_map)

In [7]:
class TfIdf:
    def __init__(self):
        self.vocab = set()
        self.df_count = {}
        self.idf = {}
        self.N = 0
    
    def tokenize(self, text):
        if not isinstance(text, str):
            return []   # ignore NaN or non-string values
        cleaned_text = re.sub(r"[^\w\s]", "", text)
        return cleaned_text.lower().split()

    def build_vocab(self, df, column='message'):
        """Build vocabulary from all documents"""
        self.vocab = set()
        
        for i in range(len(df)):
            tokens = self.tokenize(df.loc[i, column])
            self.vocab = self.vocab.union(tokens)
        
        return self.vocab
    
    def compute_tf(self, tokens):
        """Compute term frequency for a document"""
        if not tokens:
            return {}
        tf = Counter(tokens)
        total = len(tokens)
        return {word: count / total for word, count in tf.items()}

    def compute_df_count(self, df, column='message'):
        """Count document frequency for each word"""
        self.df_count = dict.fromkeys(self.vocab, 0)
        self.N = len(df)

        for i in range(len(df)):
            text = df.loc[i, column]
            tokens = set(self.tokenize(text))  # IMPORTANT: set() to count each word once per document
            for word in tokens:
                if word in self.df_count:
                    self.df_count[word] += 1
        
        return self.df_count
    
    def compute_idf(self):
        """Compute IDF values for all words in vocabulary"""
        self.idf = {}
        for word, df_val in self.df_count.items():
            self.idf[word] = np.log((self.N + 1) / (df_val + 1)) + 1
        
        return self.idf
    
    def compute_tfidf(self, tf):
        """Compute TF-IDF scores given TF values"""
        return {word: tf[word] * self.idf.get(word, 0) for word in tf}

    def compute_text_tfidf(self, text):
        """Compute TF-IDF for a single text document"""
        tokens = self.tokenize(text)
        tf = self.compute_tf(tokens)
        return self.compute_tfidf(tf)
    
    def fit(self, df, column='message'):
        """Fit the TF-IDF model on a dataframe"""
        self.build_vocab(df, column)
        self.compute_df_count(df, column)
        self.compute_idf()
        return self
    
    def transform(self, df, column='message'):
        """Transform documents to TF-IDF vectors"""
        tfidf_scores = []
        for i in range(len(df)):
            text = df.loc[i, column]
            tfidf = self.compute_text_tfidf(text)
            tfidf_scores.append(tfidf)
        return tfidf_scores
    
    def fit_transform(self, df, column='message'):
        """Fit and transform in one step"""
        self.fit(df, column)
        return self.transform(df, column)

In [8]:
# Initialize the TF-IDF model
tfidf_model = TfIdf()

# Option 1: Get dictionaries first (like your original code)
df['text_tf_idf'] = tfidf_model.fit_transform(df, column='text')

# Then vectorize (exactly like your original code)
vectorizer = DictVectorizer(sparse=False)
X_all = vectorizer.fit_transform(df['text_tf_idf'].tolist())
y_all = df['sentiment_label'].values


The data preparation is done now need to add the data to the model